In [1]:
import os
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchinfo import summary
from torch import nn    
from torch.nn import functional as F
from torch.utils.data import random_split
import numpy as np

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Model Architecture

In [3]:
class UNET(nn.Module):
    def __init__(self, in_channels=1, out_channels=2, features=(32, 64, 128, 256)):
        super().__init__()

        self.encoders = nn.ModuleList()
        current = in_channels
        for feature in features:
            self.encoders.append(self.conv_block(current, feature))
            current = feature

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = nn.Sequential(
            self.conv_block(features[-1], features[-1] * 2),
            nn.Dropout2d(0.2)
            )

        self.up_convs = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()
        
        for feature in reversed(features):
            self.up_convs.append(self.up_conv(feature * 2, feature))
            self.dec_blocks.append(self.conv_block(feature * 2, feature))

        self.final = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=out_channels),
            nn.LeakyReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=out_channels),
            nn.LeakyReLU(inplace=True),
        )

    def up_conv(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.LeakyReLU(inplace=True),
        )

    def forward(self, x):
        original = x
        skips = []

        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        for up, dec, skip in zip(self.up_convs, self.dec_blocks, reversed(skips)):
            x = up(x)           
            x = torch.cat((skip, x), dim=1)
            x = dec(x)

        mask = torch.softmax(self.final(x), dim=1)
        return mask * original

In [4]:
model = UNET(in_channels=1, out_channels=2).to(device)
summary(model, input_size=(4, 1, 1040, 224), device=device)

Layer (type:depth-idx)                   Output Shape              Param #
UNET                                     [4, 2, 1040, 224]         --
├─ModuleList: 1-7                        --                        (recursive)
│    └─Sequential: 2-1                   [4, 32, 1040, 224]        --
│    │    └─Conv2d: 3-1                  [4, 32, 1040, 224]        288
│    │    └─GroupNorm: 3-2               [4, 32, 1040, 224]        64
│    │    └─LeakyReLU: 3-3               [4, 32, 1040, 224]        --
│    │    └─Conv2d: 3-4                  [4, 32, 1040, 224]        9,216
│    │    └─GroupNorm: 3-5               [4, 32, 1040, 224]        64
│    │    └─LeakyReLU: 3-6               [4, 32, 1040, 224]        --
├─MaxPool2d: 1-2                         [4, 32, 520, 112]         --
├─ModuleList: 1-7                        --                        (recursive)
│    └─Sequential: 2-2                   [4, 64, 520, 112]         --
│    │    └─Conv2d: 3-7                  [4, 64, 520, 112]     

Data Loading

In [5]:
DATA_DIR = "../datasets/spectrograms"

In [6]:
class SpectrogramDataset(Dataset):
    def __init__(self, directory):
        self.directory = directory
        self.mixture_files = sorted([f for f in os.listdir(directory) if f.endswith('_mixture.npy')])

    def __len__(self):
        return len(self.mixture_files)

    def __getitem__(self, idx):

        mixture_path = os.path.join(self.directory, self.mixture_files[idx])

        vocal_path = mixture_path.replace('_mixture.npy', '_vocals.npy')

        instrumental_path = mixture_path.replace('_mixture.npy', '_instrumental.npy')

        mixture = np.load(mixture_path)
        vocals = np.load(vocal_path)
        instrumental = np.load(instrumental_path)

        X = torch.from_numpy(mixture).float().unsqueeze(0)

        Y = torch.from_numpy(
            np.stack([vocals, instrumental], axis=0)
        ).float()

        X = F.pad(X, (0, 8, 0, 15))
        Y = F.pad(Y, (0, 8, 0, 15))

        return X, Y

In [7]:
ds = SpectrogramDataset(DATA_DIR)

In [8]:
train_size = int(0.8 * len(ds))
val_size = len(ds) - train_size

train_dataset, val_dataset = random_split(ds, [train_size, val_size])

In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False
)

Traning Loop

In [10]:
class MRSTFTLoss(nn.Module):
    def __init__(self,fft_sizes=(64, 128, 256), hop_sizes=(16, 32, 64), win_lengths=(64, 128, 224)):
        super().__init__()

        self.fft_sizes = fft_sizes
        self.hop_sizes = hop_sizes
        self.win_lengths = win_lengths

    def forward(self, pred, target):

        B, C, H, W = pred.shape
        total_loss = 0.0

        for b in range(B):
            for c in range(C):

                pred_sample = pred[b, c]
                target_sample = target[b, c]

                for n_fft, hop, win_len in zip(self.fft_sizes, self.hop_sizes, self.win_lengths):

                    window = torch.hann_window(
                        win_len,
                        device=pred.device,
                        dtype=pred.dtype
                    )

                    pred_stft = torch.stft(
                        pred_sample,
                        n_fft=n_fft,
                        hop_length=hop,
                        win_length=win_len,
                        window=window,
                        return_complex=True,
                        pad_mode="constant"
                    ).abs()

                    target_stft = torch.stft(
                        target_sample,
                        n_fft=n_fft,
                        hop_length=hop,
                        win_length=win_len,
                        window=window,
                        return_complex=True,
                        pad_mode="constant"
                    ).abs()

                    numerator = torch.linalg.vector_norm(target_stft - pred_stft)

                    denominator = torch.linalg.vector_norm(target_stft)

                    if denominator < 1e-6:
                        sc_loss = numerator
                    else:
                        sc_loss = numerator / (denominator + 1e-8)

                    mag_loss = (torch.log(target_stft + 1e-7) - torch.log(pred_stft + 1e-7)).abs().mean()

                    total_loss += sc_loss + mag_loss

        return total_loss / (B * C * len(self.fft_sizes))

In [11]:
epochs = 150
patience = 8
lastbest = 0
best_loss_value = float("inf")

loss_fn = MRSTFTLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-6)

train_loss_batchwise = []
val_loss_batchwise = []

MODEL_DIR = "../models/"
model_name = "vocal_inst_sep_unet"

PLOT_DIR = "../plots/"
os.makedirs(MODEL_DIR, exist_ok=True)

In [12]:
def save_loss_plots():
    
    plt.plot(train_loss_batchwise)
    plt.title("Batchwise Train Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.savefig(os.path.join(PLOT_DIR, f"{model_name}_train_loss_batchwise.png"))
    plt.close()

    plt.plot(val_loss_batchwise)
    plt.title("Batchwise Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.savefig(os.path.join(PLOT_DIR, f"{model_name}_val_loss_batchwise.png"))
    plt.close()

In [ ]:
for epoch in range(epochs):

    model.train()
    running_train_loss = 0.0

    for batch_idx, (X, Y) in enumerate(train_loader):
        X = X.to(device)
        Y = Y.to(device)

        optimizer.zero_grad()

        prediction = model(X)

        loss = loss_fn(prediction,Y)

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

        if batch_idx % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] Step [{batch_idx+1}/{len(train_loader)}] Train Loss: {loss.item():.4f}")

    avg_train_loss = running_train_loss / len(train_loader)
    train_loss_batchwise.append(avg_train_loss)

    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for X, Y in val_loader:
            X = X.to(device)
            Y = Y.to(device)

            prediction = model(X)
        
            loss = loss_fn(prediction,Y)

            running_val_loss += loss.item()

    avg_val_loss = running_val_loss / len(val_loader)
    val_loss_batchwise.append(avg_val_loss)

    print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    save_loss_plots()
    
    if avg_val_loss < best_loss_value:
        best_loss_value = avg_val_loss
        lastbest = epoch

        torch.save(model.state_dict(), os.path.join(MODEL_DIR, f"{model_name}_best.pt"))
        print(f"New best model saved (val loss: {best_loss_value:.4f})")
        
    elif epoch - lastbest >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

    scheduler.step(avg_val_loss)

    
torch.save(model.state_dict(), os.path.join(MODEL_DIR, f"{model_name}_last.pt"))

Epoch [1/150] Step [1/839] Train Loss: 1.1111
Epoch [1/150] Step [11/839] Train Loss: 1.9511
Epoch [1/150] Step [21/839] Train Loss: 51.8281
Epoch [1/150] Step [31/839] Train Loss: 30.6014
Epoch [1/150] Step [41/839] Train Loss: 2.6150
Epoch [1/150] Step [51/839] Train Loss: 2.6655
Epoch [1/150] Step [61/839] Train Loss: 55.1044
Epoch [1/150] Step [71/839] Train Loss: 2.6597
Epoch [1/150] Step [81/839] Train Loss: 17.7807
Epoch [1/150] Step [91/839] Train Loss: 39.6875
Epoch [1/150] Step [101/839] Train Loss: 2.5634
Epoch [1/150] Step [111/839] Train Loss: 2.7961
Epoch [1/150] Step [121/839] Train Loss: 15.0001
Epoch [1/150] Step [131/839] Train Loss: 4.3065
Epoch [1/150] Step [141/839] Train Loss: 5.9482
Epoch [1/150] Step [151/839] Train Loss: 32.7903
Epoch [1/150] Step [161/839] Train Loss: 2.5752
Epoch [1/150] Step [171/839] Train Loss: 4.9126
Epoch [1/150] Step [181/839] Train Loss: 6.2233
Epoch [1/150] Step [191/839] Train Loss: 1.8688
Epoch [1/150] Step [201/839] Train Loss: 8.1

In [ ]:
model = UNET(in_channels=1, out_channels=2)

In [ ]:
state_dict = torch.load("../models/vocal_inst_sep_unet_best.pt", map_location=device)
model.load_state_dict(state_dict)
model.eval()

dummy_input = torch.randn(1, 1, 1040, 256)

torch.onnx.export(model, dummy_input, "../models/vocal_inst_sep.onnx", dynamo=False)

C:\Users\vallu\AppData\Local\Temp\ipykernel_25484\2736576058.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy_input, "../models/vocal_inst_sep.onnx", dynamo=False)
